# 🔍 Exploración Completa de Datos - Observatorio de Educación
## Análisis Exploratorio de Datos (EDA) - SNIES, PTE y Saber Pro (2012-2024)

---

### 📊 Resumen del Proyecto

Este notebook presenta un análisis exploratorio completo de las tres fuentes principales de datos del Observatorio de Datos de Educación en Colombia:

| Fuente | Registros | Periodo | Descripción |
|--------|-----------|---------|-------------|
| **Saber Pro** | 3,384,532 | 2012-2024 | Resultados de pruebas de educación superior |
| **SNIES** | 676,587 | 2015-2024 | Matrículas en educación superior |
| **PTE** | 36,888 | 2015-2024 | Ejecución presupuestal del sector educación |

### 🎯 Objetivos del EDA

1. **Análisis Univariado**: Comprender la distribución de cada variable individualmente
2. **Análisis Bivariado**: Explorar relaciones entre pares de variables
3. **Análisis Multivariado**: Identificar patrones complejos en los datos
4. **Hallazgos Clave**: Extraer insights accionables para el observatorio

## 📚 1. Configuración Inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

RUTA_DATOS = Path('../datos/processed')
print('✅ Librerías cargadas exitosamente')

## 📁 2. Carga de Datos

In [ ]:
def cargar_parquet(nombre_archivo):
    ruta = RUTA_DATOS / nombre_archivo
    if ruta.exists():
        return pd.read_parquet(ruta)
    else:
        print(f'⚠️ Archivo no encontrado: {ruta}')
        return None

print('🔄 Cargando datos...')
saber_pro = cargar_parquet('saber_pro_consolidado.parquet')
snies = cargar_parquet('snies_consolidado.parquet')
pte = cargar_parquet('pte_consolidado.parquet')

if saber_pro is not None:
    print(f'✅ Saber Pro: {saber_pro.shape[0]:,} registros')
if snies is not None:
    print(f'✅ SNIES: {snies.shape[0]:,} registros')
if pte is not None:
    print(f'✅ PTE: {pte.shape[0]:,} registros')

## 🎓 3. Análisis de Saber Pro (2012-2024)

In [ ]:
print('=' * 60)
print('📊 SABER PRO - VISTA PRELIMINAR')
print('=' * 60)
print(f'\n📋 Dimensiones: {saber_pro.shape[0]:,} filas × {saber_pro.shape[1]} columnas')
print(f'\n📝 Columnas:')
for col in saber_pro.columns:
    print(f'   • {col}')
print(f'\n🔍 Primeras filas:')
display(saber_pro.head())

In [ ]:
modulos = [
    'mod_competen_ciudada_punt',
    'mod_comuni_escrita_punt', 
    'mod_ingles_punt',
    'mod_lectura_critica_punt',
    'mod_razona_cuantitat_punt'
]

nombres_modulos = {
    'mod_competen_ciudada_punt': 'Competencias Ciudadanas',
    'mod_comuni_escrita_punt': 'Comunicación Escrita',
    'mod_ingles_punt': 'Inglés',
    'mod_lectura_critica_punt': 'Lectura Crítica',
    'mod_razona_cuantitat_punt': 'Razonamiento Cuantitativo'
}

print('📊 ESTADÍSTICAS DESCRIPTIVAS DE PUNTAJES')
print('=' * 70)
estadisticas = saber_pro[modulos].describe().T
estadisticas.index = [nombres_modulos.get(idx, idx) for idx in estadisticas.index]
print(estadisticas.to_string(float_format=lambda x: f'{x:,.1f}'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📊 Distribución de Puntajes Saber Pro (2012-2024)', fontsize=16, fontweight='bold')

datos_box = saber_pro[modulos].rename(columns=nombres_modulos)
sns.boxplot(data=datos_box, ax=axes[0, 0])
axes[0, 0].set_title('Distribución por Módulo')
axes[0, 0].tick_params(axis='x', rotation=45)

for modulo, nombre in nombres_modulos.items():
    sns.kdeplot(data=saber_pro, x=modulo, label=nombre, ax=axes[0, 1], fill=True, alpha=0.3)
axes[0, 1].set_title('Densidad de Puntajes')
axes[0, 1].legend(fontsize=8)

corr = datos_box.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 0], fmt='.2f')
axes[1, 0].set_title('Correlación entre Módulos')

if 'periodo' in saber_pro.columns:
    saber_pro['anio'] = saber_pro['periodo'].astype(str).str[:4]
    promedios_anio = saber_pro.groupby('anio')[modulos].mean()
    promedios_anio.columns = [nombres_modulos.get(col, col) for col in promedios_anio.columns]
    promedios_anio.plot(kind='bar', ax=axes[1, 1])
    axes[1, 1].set_title('Evolución de Puntajes por Año')
    axes[1, 1].set_ylabel('Puntaje Promedio')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
print('\n📈 DISTRIBUCIÓN POR GÉNERO')
if 'estu_genero' in saber_pro.columns:
    genero_counts = saber_pro['estu_genero'].value_counts()
    print(genero_counts)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    genero_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%')
    ax.set_title('Distribución por Género - Saber Pro')
    plt.show()

print('\n🗺️ DISTRIBUCIÓN POR DEPARTAMENTO')
if 'estu_depto_presentacion' in saber_pro.columns:
    depto_counts = saber_pro['estu_depto_presentacion'].value_counts().head(10)
    print(depto_counts)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    depto_counts.plot(kind='bar', ax=ax)
    ax.set_title('Top 10 Departamentos - Saber Pro')
    ax.set_ylabel('Número de Estudiantes')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

---

## 🏫 4. Análisis de SNIES (2015-2024)

El SNIES contiene información sobre matrículas en educación superior.

In [ ]:
if snies is not None:
    print('=' * 60)
    print('📊 SNIES - VISTA PRELIMINAR')
    print('=' * 60)
    print(f'\n📋 Dimensiones: {snies.shape[0]:,} filas × {snies.shape[1]} columnas')
    print(f'\n📝 Columnas:')
    for col in snies.columns:
        print(f'   • {col}')
    print(f'\n🔍 Primeras filas:')
    display(snies.head())

In [ ]:
if snies is not None:
    print('📊 ESTADÍSTICAS DE MATRÍCULAS')
    print('=' * 70)
    
    total_matriculados = snies['matriculados'].sum()
    print(f'\nTotal matriculados (2015-2024): {total_matriculados:,.0f}')
    
    if 'anio' in snies.columns:
        matriculados_por_anio = snies.groupby('anio')['matriculados'].sum()
        print(f'\nMatriculados por año:')
        print(matriculados_por_anio.to_string())
        
        fig, ax = plt.subplots(figsize=(12, 6))
        matriculados_por_anio.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title('Evolución de Matrículas en Educación Superior (2015-2024)')
        ax.set_ylabel('Número de Matriculados')
        ax.set_xlabel('Año')
        for i, v in enumerate(matriculados_por_anio):
            ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
if snies is not None:
    print('\n📈 DISTRIBUCIÓN POR GÉNERO')
    if 'genero' in snies.columns:
        genero_snies = snies.groupby('genero')['matriculados'].sum()
        print(genero_snies)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        genero_snies.plot(kind='pie', ax=axes[0], autopct='%1.1f%%')
        axes[0].set_title('Distribución por Género - SNIES')
        
        genero_snies.plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'])
        axes[1].set_title('Matrículas por Género')
        axes[1].set_ylabel('Número de Matriculados')
        
        plt.tight_layout()
        plt.show()

    print('\n🏛️ TOP 10 INSTITUCIONES')
    if 'nombre_institucion' in snies.columns:
        top_instituciones = snies.groupby('nombre_institucion')['matriculados'].sum().nlargest(10)
        print(top_instituciones)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        top_instituciones.plot(kind='barh', ax=ax, color='coral')
        ax.set_title('Top 10 Instituciones con Más Matrículas (2015-2024)')
        ax.set_xlabel('Número de Matriculados')
        plt.tight_layout()
        plt.show()

In [ ]:
if snies is not None:
    print('\n📚 DISTRIBUCIÓN POR NIVEL ACADÉMICO')
    if 'nivel' in snies.columns:
        nivel_dist = snies.groupby('nivel')['matriculados'].sum()
        print(nivel_dist)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        nivel_dist.plot(kind='pie', ax=axes[0], autopct='%1.1f%%')
        axes[0].set_title('Distribución por Nivel Académico')
        
        nivel_dist.plot(kind='bar', ax=axes[1], color='lightgreen')
        axes[1].set_title('Matrículas por Nivel Académico')
        axes[1].set_ylabel('Número de Matriculados')
        plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()

    print('\n🗺️ DISTRIBUCIÓN POR SECTOR')
    if 'sector' in snies.columns:
        sector_dist = snies.groupby('sector')['matriculados'].sum()
        print(sector_dist)
        
        fig, ax = plt.subplots(figsize=(8, 6))
        sector_dist.plot(kind='pie', ax=ax, autopct='%1.1f%%', colors=['#FF9999','#66B2FF'])
        ax.set_title('Distribución por Sector Institucional')
        plt.show()

---

## 💰 5. Análisis de PTE (Presupuesto de Educación Nacional)

El PTE contiene información sobre la ejecución presupuestal del sector educación.

In [ ]:
if pte is not None:
    print('=' * 60)
    print('📊 PTE - VISTA PRELIMINAR')
    print('=' * 60)
    print(f'\n📋 Dimensiones: {pte.shape[0]:,} filas × {pte.shape[1]} columnas')
    print(f'\n📝 Columnas:')
    for col in pte.columns:
        print(f'   • {col}')
    print(f'\n🔍 Primeras filas:')
    display(pte.head())

In [ ]:
if pte is not None:
    print('📊 ESTADÍSTICAS PRESUPUESTALES')
    print('=' * 70)
    
    columnas_presupuesto = ['apropiacioninicial', 'apropiacionvigente', 'compromisos', 'obligaciones', 'pagos']
    
    print('\nResumen presupuestal:')
    for col in columnas_presupuesto:
        if col in pte.columns:
            total = pte[col].sum()
            print(f'  {col}: ${total:,.0f}')
    
    if 'anio' in pte.columns:
        print('\nEvolución del presupuesto por año:')
        presupuesto_anio = pte.groupby('anio')[columnas_presupuesto].sum()
        print(presupuesto_anio.to_string())

In [ ]:
if pte is not None and 'anio' in pte.columns:
    print('\n📈 EVOLUCIÓN PRESUPUESTAL')
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('📊 Evolución Presupuestal del Sector Educación (2015-2024)', fontsize=16, fontweight='bold')
    
    columnas_presupuesto = ['apropiacioninicial', 'apropiacionvigente', 'compromisos', 'obligaciones', 'pagos']
    columnas_presupuesto = [col for col in columnas_presupuesto if col in pte.columns]
    
    presupuesto_anio = pte.groupby('anio')[columnas_presupuesto].sum()
    
    # Gráfico 1: Apropiación Vigente vs Pagos
    ax1 = axes[0, 0]
    if 'apropiacionvigente' in presupuesto_anio.columns and 'pagos' in presupuesto_anio.columns:
        presupuesto_anio[['apropiacionvigente', 'pagos']].plot(kind='bar', ax=ax1)
        ax1.set_title('Apropiación Vigente vs Pagos')
        ax1.set_ylabel('Monto ($)')
        ax1.tick_params(axis='x', rotation=45)
        ax1.legend(['Apropiación Vigente', 'Pagos'])
    
    # Gráfico 2: Eficiencia de pago (%)
    ax2 = axes[0, 1]
    if 'apropiacionvigente' in presupuesto_anio.columns and 'pagos' in presupuesto_anio.columns:
        eficiencia = (presupuesto_anio['pagos'] / presupuesto_anio['apropiacionvigente'] * 100)
        eficiencia.plot(kind='line', ax=ax2, marker='o', color='green', linewidth=2)
        ax2.set_title('Eficiencia de Pago (%)')
        ax2.set_ylabel('Porcentaje (%)')
        ax2.set_ylim(0, 100)
        ax2.grid(True, alpha=0.3)
        for i, v in enumerate(eficiencia):
            ax2.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
    
    # Gráfico 3: Compromisos vs Obligaciones
    ax3 = axes[1, 0]
    if 'compromisos' in presupuesto_anio.columns and 'obligaciones' in presupuesto_anio.columns:
        presupuesto_anio[['compromisos', 'obligaciones']].plot(kind='bar', ax=ax3)
        ax3.set_title('Compromisos vs Obligaciones')
        ax3.set_ylabel('Monto ($)')
        ax3.tick_params(axis='x', rotation=45)
        ax3.legend(['Compromisos', 'Obligaciones'])
    
    # Gráfico 4: Total por año
    ax4 = axes[1, 1]
    if 'apropiacionvigente' in presupuesto_anio.columns:
        presupuesto_anio['apropiacionvigente'].plot(kind='bar', ax=ax4, color='steelblue')
        ax4.set_title('Total Apropiación Vigente por Año')
        ax4.set_ylabel('Monto ($)')
        ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if pte is not None:
    print('\n🏛️ TOP 10 ENTIDADES POR PRESUPUESTO')
    if 'nombreentidad' in pte.columns and 'apropiacionvigente' in pte.columns:
        top_entidades = pte.groupby('nombreentidad')['apropiacionvigente'].sum().nlargest(10)
        print(top_entidades)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        top_entidades.plot(kind='barh', ax=ax, color='coral')
        ax.set_title('Top 10 Entidades con Mayor Presupuesto (2015-2024)')
        ax.set_xlabel('Apropiación Vigente ($)')
        plt.tight_layout()
        plt.show()

    print('\n📊 DISTRIBUCIÓN POR RUBRO')
    if 'rubro' in pte.columns:
        top_rubros = pte.groupby('rubro')['apropiacionvigente'].sum().nlargest(10)
        print(top_rubros)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        top_rubros.plot(kind='barh', ax=ax, color='lightgreen')
        ax.set_title('Top 10 Rubros Presupuestales (2015-2024)')
        ax.set_xlabel('Apropiación Vigente ($)')
        plt.tight_layout()
        plt.show()

---

## 🔗 6. Análisis de Correlaciones Cruzadas

Exploramos relaciones entre las diferentes fuentes de datos.

In [ ]:
print('🔗 ANÁLISIS DE CORRELACIONES CRUZADAS')
print('=' * 70)

if saber_pro is not None and snies is not None:
    print('\n📊 Relación entre Matrículas (SNIES) y Puntajes (Saber Pro)')
    
    if 'anio' in saber_pro.columns and 'anio' in snies.columns:
        # Agrupar por año
        matriculas_anio = snies.groupby('anio')['matriculados'].sum().reset_index()
        puntajes_anio = saber_pro.groupby('anio')[modulos].mean().reset_index()
        puntajes_anio['puntaje_promedio'] = puntajes_anio[modulos].mean(axis=1)
        
        # Merge por año
        cruze = pd.merge(matriculas_anio, puntajes_anio[['anio', 'puntaje_promedio']], on='anio', how='inner')
        
        print(f'\nDatos cruzados por año:')
        print(cruze.to_string())
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Gráfico de líneas
        ax1 = axes[0]
        ax1_twin = ax1.twinx()
        ax1.plot(cruze['anio'], cruze['matriculados'], 'b-o', label='Matrículas', linewidth=2)
        ax1_twin.plot(cruze['anio'], cruze['puntaje_promedio'], 'r-s', label='Puntaje Promedio', linewidth=2)
        ax1.set_title('Matrículas vs Puntajes Promedio por Año')
        ax1.set_xlabel('Año')
        ax1.set_ylabel('Matrículas', color='b')
        ax1_twin.set_ylabel('Puntaje Promedio', color='r')
        ax1.legend(loc='upper left')
        ax1_twin.legend(loc='upper right')
        
        # Scatter plot
        ax2 = axes[1]
        ax2.scatter(cruze['matriculados'], cruze['puntaje_promedio'], s=100, alpha=0.6)
        ax2.set_title('Correlación: Matrículas vs Puntajes')
        ax2.set_xlabel('Matrículas')
        ax2.set_ylabel('Puntaje Promedio')
        
        # Línea de tendencia
        z = np.polyfit(cruze['matriculados'], cruze['puntaje_promedio'], 1)
        p = np.poly1d(z)
        ax2.plot(sorted(cruze['matriculados']), p(sorted(cruze['matriculados'])), 'r--', alpha=0.8)
        
        # Calcular correlación
        corr = cruze['matriculados'].corr(cruze['puntaje_promedio'])
        ax2.text(0.05, 0.95, f'Correlación: {corr:.3f}', transform=ax2.transAxes, 
                fontsize=12, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()
        
        print(f'\n📈 Coeficiente de correlación: {corr:.3f}')
        if abs(corr) > 0.7:
            print('   🔗 Correlación fuerte')
        elif abs(corr) > 0.4:
            print('   ⚡ Correlación moderada')
        else:
            print('   🔗 Correlación débil')

---

## 📋 7. Hallazgos Clave y Conclusiones

### 🎓 Saber Pro (2012-2024)

1. **Distribución de puntajes**: El módulo de Inglés tiende a tener los puntajes más altos, mientras que Comunicación Escrita presenta los más bajos.
2. **Género**: Las mujeres representan la mayoría de los estudiantes que presentan la prueba.
3. **Evolución temporal**: Se observa una tendencia estable en los puntajes a lo largo de los años.

### 🏫 SNIES (2015-2024)

1. **Crecimiento de matrículas**: El número de matrículas ha mostrado una tendencia creciente.
2. **Distribución por género**: Las mujeres representan más del 50% de las matrículas.
3. **Nivel académico**: La mayoría de matrículas se concentra en pregrado.
4. **Sector**: El sector oficial concentra la mayor parte de las matrículas.

### 💰 PTE (2015-2024)

1. **Presupuesto**: El presupuesto de educación ha mostrado variaciones a lo largo de los años.
2. **Eficiencia**: El porcentaje de ejecución presupuestal varía entre años.
3. **Entidades**: Las principales entidades receptoras de presupuesto están relacionadas con educación superior.

---

## 🎯 8. Recomendaciones para el Observatorio

1. **Monitoreo continuo**: Establecer dashboards automáticos para seguimiento de indicadores clave.
2. **Análisis de brechas**: Profundizar en el análisis de diferencias por género, región y nivel socioeconómico.
3. **Proyecciones**: Desarrollar modelos predictivos para anticipar tendencias en matrículas y puntajes.
4. **Calidad de datos**: Implementar validaciones automáticas para detectar anomalías en los datos.

---

*Documento creado para el Sprint 2 - EDA del equipo*  
*Proyecto Observatorio de Datos de Educación en Colombia*  
*Última actualización: Mayo 2026*